# 03 · Ingestão Bronze via Auto Loader

Lê os arquivos Parquet do diretório de landing
(`/Volumes/antecipeai/landing/raw/incidentes/`) via **Auto Loader**
(`cloudFiles`), que faz leitura incremental — só processa arquivos novos
desde a última execução, controlado por checkpoint.

**Bronze = "na íntegra"**: nenhum tratamento de tipo, nenhuma limpeza.
Os dados já chegam como string (decisão tomada no notebook `02`), e aqui
só adicionamos metadata técnica de ingestão:
- `_ingested_at`: timestamp de quando a linha entrou na Bronze
- `_source_file`: arquivo de origem de cada linha (rastreabilidade)

Usamos `.trigger(availableNow=True)`: processa tudo que estiver disponível
agora e para — sem manter um cluster rodando 24/7 (importante pro custo
zero do Databricks Free). Quando novos arquivos chegarem na landing,
basta rodar este notebook de novo.

In [0]:
%run ./00_config

## Paths de checkpoint, schema location e origem/destino

In [0]:
from pyspark.sql import functions as F

source_path = volume_path("incidentes")
checkpoint_path = volume_path("_checkpoints/bronze_incidentes")
schema_location = volume_path("_schema/bronze_incidentes")
bronze_table = qualified_table(SCHEMA_BRONZE, "incidentes")

print("Origem (landing):     ", source_path)
print("Checkpoint:            ", checkpoint_path)
print("Schema location:       ", schema_location)
print("Tabela destino (Bronze):", bronze_table)

## Column Mapping — criar a tabela vazia antes do Auto Loader

Os nomes de coluna vêm exatamente como no arquivo de origem (`Grupo
designado`, `Descrição resumida`, `Entrou para KPI?`...), com espaços e
caracteres que o Delta rejeita por padrão
(`DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES`). Para manter os nomes
originais "na íntegra" — como a Bronze exige — sem sanitizar/renomear
nessa camada (isso fica pra Silver), precisamos habilitar Column
Mapping **antes** da tabela ser criada pela primeira vez.

Duas abordagens foram tentadas e **não funcionam neste ambiente**
(Databricks Free / compute serverless), documentando aqui pra não
repetir o erro:
- `.option("delta.columnMapping.mode", "name")` no `writeStream`: não
aplica propriedade de tabela na criação via streaming.
- `spark.conf.set("spark.databricks.delta.properties.defaults...")`:
falha com `CONFIG_NOT_AVAILABLE` — serverless restringe quais
configurações internas podem ser setadas em runtime.

**O que funciona:** criar a tabela **vazia** primeiro via
`DataFrameWriterV2` (`.writeTo(...).tableProperty(...)`), que tem
suporte de verdade a propriedades de tabela na criação — sem depender de
`spark.conf` nem do `.option()` do streaming. O schema é descoberto com
uma leitura batch (`.limit(0)`, sem processar nenhuma linha de verdade)
dos mesmos arquivos que o Auto Loader vai ler depois — schema-on-read
continua intacto, nada é hardcoded na mão.

In [0]:
if not spark.catalog.tableExists(bronze_table):
    schema_probe = (
        spark.read.format("parquet")
        .load(source_path)
        .limit(0)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        # O Auto Loader sempre adiciona essa coluna quando schema evolution
        # está ativo (guarda dado que não bateu com o schema esperado) —
        # precisa já existir na tabela, senão dá DELTA_METADATA_MISMATCH no
        # primeiro micro-batch, e o Unity Catalog bloqueia merge automático
        # de schema em tabelas com Table ACLs habilitadas.
        .withColumn("_rescued_data", F.lit(None).cast("string"))
    )

    (
        schema_probe.writeTo(bronze_table)
        .using("delta")
        .tableProperty("delta.columnMapping.mode", "name")
        .tableProperty("delta.minReaderVersion", "2")
        .tableProperty("delta.minWriterVersion", "5")
        .create()
    )
    print(f"Tabela {bronze_table} criada vazia, já com Column Mapping habilitado.")
else:
    print(f"Tabela {bronze_table} já existe — criação pulada.")
    print("Se ela foi criada ANTES desta correção (sem Column Mapping), use o widget de reset abaixo.")

## Reset opcional da tabela Bronze

Se a tabela `bronze.incidentes` já existe de uma tentativa anterior
**sem** Column Mapping habilitado, a checagem `tableExists` acima pula a
criação e o problema persiste. Este widget existe pra isso: fica
desligado por padrão (não apaga nada sozinho ao rodar o notebook
inteiro de novo), e só derruba a tabela + limpa o checkpoint/schema
location do Auto Loader quando você explicitamente muda pra `true`.
**Depois de resetar, volte e rode a célula acima de novo** antes de
seguir para o Auto Loader.

In [0]:
dbutils.widgets.dropdown("reset_bronze_table", "false", ["false", "true"], "Derrubar bronze.incidentes e recriar do zero?")

if dbutils.widgets.get("reset_bronze_table") == "true":
    spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
    dbutils.fs.rm(checkpoint_path, recurse=True)
    dbutils.fs.rm(schema_location, recurse=True)
    print(f"Reset concluído: {bronze_table} derrubada, checkpoint e schema location limpos.")
    print("Volte para a célula de criação da tabela (acima) e rode de novo.")
else:
    print("Reset não solicitado (reset_bronze_table=false) — nada foi apagado.")

## Leitura incremental com Auto Loader

`cloudFiles.inferColumnTypes` fica `false` de propósito — mesmo o
Parquet de origem já vindo 100% string (garantido no notebook `02`),
deixamos explícito aqui que a Bronze nunca deve inferir tipos. Isso é o
que torna essa camada resiliente a mudanças de schema na origem: uma
coluna nova na próxima extração da Locaweb é automaticamente capturada
(schema evolution do Auto Loader), sem quebrar o job.

In [0]:
raw_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

bronze_stream = (
    raw_stream
    .withColumn("_ingested_at", F.current_timestamp())
    # _metadata.file_path (coluna oculta do Spark, exposta automaticamente em
    # toda leitura de arquivo) — NÃO usar F.input_file_name(): o Unity
    # Catalog bloqueia essa função legada
    # ([UC_COMMAND_NOT_SUPPORTED.WITH_RECOMMENDATION]), já que a linhagem de
    # arquivo passa a ser controlada pelo próprio UC. É Spark padrão desde a
    # 3.4 (SPARK-37273), não é exclusividade Databricks.
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

## Escrita incremental na tabela Delta da Bronze

In [0]:
query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print(f"Ingestão concluída. Linhas na Bronze agora: {spark.table(bronze_table).count()}")

## Conferência

In [0]:
df_bronze = spark.table(bronze_table)
df_bronze.printSchema()
display(df_bronze.limit(10))

In [0]:
# Resultado esperado (validado com o dataset real fora do Databricks antes de
# escrever este notebook): 122.543 linhas, 19 colunas originais + as 2 colunas
# de metadata (_ingested_at, _source_file, _rescued_data) = 22 colunas. Todas as 19 colunas
# originais devem aparecer como StringType — se alguma vier tipada diferente,
# a etapa de conversão do notebook 02 não rodou como esperado.